# Applied Econometric Examples & Sanity Checking
This notebook demonstrates how the `stats-transformer` models compare against native library baselines (like `statsmodels`) using classic econometric datasets. This serves as a sanity check to ensure the wrapper logic computes identical results.

## 1. The Grunfeld Investment Panel Data
We will pull the classic Grunfeld investment dataset and fit a Panel Fixed Effects (LSDV - Least Squares Dummy Variable) model.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from stats_transformer.models.regression.regression import RegressionModel

# 1. Fetch Data
print("Fetching Grunfeld Investment Panel Data...")
df_raw = sm.datasets.grunfeld.load_pandas().data
display(df_raw.head())

Fetching Grunfeld Investment Panel Data...


,invest,value,capital,firm,year
0,317.6,3078.5,2.8,General Motors,1935.0
1,391.8,4661.7,52.6,General Motors,1936.0
2,410.6,5387.1,156.9,General Motors,1937.0
3,257.7,2792.2,209.2,General Motors,1938.0
4,330.8,4313.2,203.4,General Motors,1939.0


## 2. Baseline: Native Statsmodels (LSDV)
We compute the baseline using `statsmodels` directly, adding dummy variables for the entity (firm) to replicate fixed effects.

In [2]:
y = df_raw["invest"]
# Include firm as categorical for fixed effects (Dummy Variables), without intercept
X = df_raw[["value", "capital"]].copy()
entity_dummies = pd.get_dummies(df_raw["firm"], prefix="entity", dtype=int)
X = pd.concat([X, entity_dummies], axis=1)

sm_model = sm.OLS(y, X).fit()
print(sm_model.summary().tables[1])

                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
value                        0.1101      0.011      9.746      0.000       0.088       0.132
capital                      0.3100      0.017     18.744      0.000       0.277       0.343
entity_American Steel      -20.5782     11.298     -1.821      0.070     -42.852       1.695
entity_Atlantic Refining  -114.6025     13.502     -8.488      0.000    -141.222     -87.983
entity_Chrysler            -27.8091     13.419     -2.072      0.039     -54.264      -1.355
entity_Diamond Match        -6.5680     11.274     -0.583      0.561     -28.794      15.658
entity_General Electric   -235.5694     23.286    -10.116      0.000    -281.478    -189.661
entity_General Motors      -70.2991     47.375     -1.484      0.139    -163.699      23.101
entity_Goodyear            -87.2145     12.289     -7.097      0.000  

## 3. Stats-Transformer: RegressionModel with Fixed Effects
Now we run the exact same regression using the `stats-transformer` object-oriented wrapper, enabling `add_entity_fixed_effects`.

In [6]:
st_model = RegressionModel(
    target="invest",
    independent_variables=["value", "capital"],
    add_entity_fixed_effects=True,
    entity_column="firm"
)
st_model.fit(df_raw)
print(st_model.get_summary())

                            OLS Regression Results                            
Dep. Variable:                 invest   R-squared:                       0.946
Model:                            OLS   Adj. R-squared:                  0.943
Method:                 Least Squares   F-statistic:                     302.6
Date:                Fri, 08 May 2026   Prob (F-statistic):          4.77e-124
Time:                        10:45:39   Log-Likelihood:                -1167.4
No. Observations:                 220   AIC:                             2361.
Df Residuals:                     207   BIC:                             2405.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
value                   

## 4. Sanity Check Comparison
Finally, we extract the coefficients and standard errors from both implementations and assert that they are mathematically identical.

In [4]:
# Align indexes as column order might differ
sm_params = sm_model.params.sort_index()
st_params = st_model.model.params.sort_index()

sm_bse = sm_model.bse.sort_index()
st_bse = st_model.model.bse.sort_index()

param_diff = np.max(np.abs(sm_params.values - st_params.values))
bse_diff = np.max(np.abs(sm_bse.values - st_bse.values))

print(f"Max difference in Coefficients: {param_diff:.6e}")
print(f"Max difference in Standard Errors: {bse_diff:.6e}")

if param_diff < 1e-10 and bse_diff < 1e-10:
    print("\n✅ SUCCESS: Stats-Transformer perfectly matches native statsmodels Fixed Effects (LSDV) regression.")
else:
    print("\n❌ FAILURE: Significant deviations detected.")
    assert False, "Sanity check failed!"

Max difference in Coefficients: 0.000000e+00
Max difference in Standard Errors: 0.000000e+00

✅ SUCCESS: Stats-Transformer perfectly matches native statsmodels Fixed Effects (LSDV) regression.
